In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import optuna
from sklearn.model_selection import cross_val_score
from category_encoders import TargetEncoder
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

C:\Users\Sofia\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025newbig2.csv')
df_cheap = df[df['Цена'] <= 700_000]
df_mid = df[(df['Цена'] >= 500_000) & (df['Цена'] < 2_000_000)]
df_exp = df[df['Цена'] >= 2_000_000]
df_mid.head()

,Название машины,Год,Цена,Объем двигателя,Тип двигателя,Мощность,Коробка передач,Привод,Пробег,Руль,Поколение,Рестайлинг,Тип кузова,Метка,Город,Год объявления,Месяц объявления,Возраст авто
3,Nissan Tiida Latio,2009.0,660000.0,1.5,бензин,109.0,CVT,передний,168000.0,правый,1.0,1.0,седан,nissan,Лесозаводск,2024,9,15.0
4,Honda Fit Shuttle,2012.0,600000.0,1.3,бензин,88.0,CVT,передний,158059.0,правый,1.0,0.0,универсал,honda,Владивосток,2021,2,9.0
5,Renault Duster,2017.0,820000.0,1.5,дизель,109.0,МКПП,4WD,95000.0,левый,1.0,1.0,джип/suv 5 дв.,renault,Норильск,2021,5,4.0
6,Ford C-MAX,2008.0,540000.0,1.8,бензин,125.0,МКПП,передний,201000.0,левый,1.0,1.0,минивэн,ford,Омск,2023,5,15.0
10,BMW 5-Series,2013.0,1050000.0,2.0,бензин,184.0,АКПП,задний,199000.0,левый,6.0,0.0,седан,bmw,Москва,2021,5,8.0


In [3]:
df_mid = df_mid.drop(columns=[
    'Название машины',
    'Месяц объявления',
    'Руль',
    'Рестайлинг'
])


In [4]:
categorical_target = ['Метка', 'Город', 'Поколение']      # Target Encoding
categorical_onehot = ['Тип двигателя', 'Коробка передач', 'Привод', 'Тип кузова']  # OneHot
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Возраст авто']
categories = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Тип кузова', 'Метка', 'Город', 'Руль']

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(), categorical_target),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_onehot),
        ('num', 'passthrough', numerical)
    ]
)

In [6]:
y = df_mid['Цена']
X = df_mid.drop('Цена', axis=1)

X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
X_train = X_train.drop(columns=['Год объявления'])
X_test = X_test.drop(columns=['Год объявления'])
y_train = y_train.drop(columns=['Год объявления'])
y_test = y_test.drop(columns=['Год объявления'])

In [7]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=2000,
        n_jobs=-1,
        random_state=42,
        eval_metric='rmse'
    ))
])
model.fit(X_train, y_train, regressor__verbose=True)
y_pred = model.predict(X_test)

In [8]:
xg_mse = mean_squared_error(y_test, y_pred)
xg_rmse = np.sqrt(xg_mse)
xg_mae = mean_absolute_error(y_test, y_pred)
xg_r2 = r2_score(y_test, y_pred)


In [9]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [xg_mse, xg_rmse, xg_mae, xg_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),39_884_003_782.94
1,Среднеквадратическая ошибка (RMSE),199_709.80
2,Средняя абсолютная ошибка (MAE),149_568.92
3,Коэффицент детерминации (R^2),0.74


In [10]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.2%}")

MAPE: 14.35%


In [11]:
def objective_gpu(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "gamma": trial.suggest_float("gamma", 0, 3),
    }

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor(
            **params,
            device="cuda",           # GPU
            tree_method="hist",
            eval_metric="rmse",
            random_state=42
        ))
    ])

    score = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=3,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

In [12]:
study = optuna.create_study(
    study_name="xgb_gpu_car_price",
    direction="maximize"
)

study.optimize(
    objective_gpu,
    n_trials=50,
    show_progress_bar=True
)

print("Лучшие параметры:", study.best_params)

In [13]:
best_params = study.best_params

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        **best_params,
        device="cuda",
        tree_method="hist",
        eval_metric="rmse",
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

C:\Users\Sofia\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\core.py:751: UserWarning: [15:13:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [14]:
xg_mse = mean_squared_error(y_test, y_pred)
xg_rmse = np.sqrt(xg_mse)
xg_mae = mean_absolute_error(y_test, y_pred)
xg_r2 = r2_score(y_test, y_pred)

In [15]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)',
                     'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [xg_mse, xg_rmse, xg_mae, xg_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),44_083_781_891.64
1,Среднеквадратическая ошибка (RMSE),209_961.38
2,Средняя абсолютная ошибка (MAE),159_086.32
3,Коэффицент детерминации (R^2),0.72


In [16]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.2%}")

MAPE: 14.90%


In [17]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.Series(result.importances_mean, index=X_test.columns)
print(importance.sort_values(ascending=False))

Год               1.86
Возраст авто      0.30
Мощность          0.29
Метка             0.22
Тип кузова        0.17
Объем двигателя   0.17
Поколение         0.07
Пробег            0.06
Коробка передач   0.03
Привод            0.02
Город             0.01
Тип двигателя     0.01
dtype: float64
